# PanTS SegResNet on Colab Pro — random vs SuPreM initialization

**This notebook orchestrates. It contains no science.** Every model, transform,
sampler, loss and checkpoint lives in `src/` in the pinned Git commit, so what
runs here is exactly what runs on the laptop.

The controlled ablation is two runs that differ in **one** argument:

| | arm A | arm B |
|---|---|---|
| name | SegResNet-Random | SegResNet initialized from SuPreM supervised pretraining |
| `--initialization` | `random` | `suprem` |
| everything else | identical | identical |

Prepared data, fold, seed, batch size, sampling, augmentation, DiceCE loss,
optimizer, schedule, AMP, step count and validation procedure are shared.

**Data flow.** Google Drive is *persistent transport*. `/content` is the
*ephemeral fast disk* training actually reads. Nothing trains from
`/content/drive`: a FUSE mount cannot sustain random reads of thousands of
files per epoch.

**Run order:** 1 → 8 once per session, then 9 (calibration) or 10 (production).
Section 11 is the disconnect-recovery path.

PanTS-te is never read here.

## 0. The only thing you edit

Pin the exact commit. A branch name would silently change what you ran.

In [ ]:
# The immutable tag created for this milestone. A tag is reproducible in a way a
# branch name is not: `agent/local-preprocess-colab` moves with every commit,
# this does not. Replace with a raw SHA only if you need a different revision.
PINNED_COMMIT = "segresnet-colab-v1"
REPO_URL      = "https://github.com/sabinthapa100/pants_sabin.git"

DRIVE_DATA    = "/content/drive/MyDrive/PanTS_prepared/segresnet"
DRIVE_RUNS    = "/content/drive/MyDrive/PanTS_runs"

REPO_DIR      = "/content/pants_sabin"
PREPARED_ROOT = "/content/PanTS_prepared"
RUNS_LOCAL    = "/content/runs"
CHECKPOINT    = "/content/pretrained/supervised_suprem_segresnet_2100.pth"

# Independently measured from the official release; the notebook refuses to
# proceed if the download does not match.
SUPREM_SHA256 = "2db81dc05cd9ea7234ca75e921e53e32b8716dc4cba88a6710742bfc282589a3"
SUPREM_URL    = ("https://huggingface.co/MrGiovanni/SuPreM/resolve/main/"
                 "supervised_suprem_segresnet_2100.pth?download=true")
print("pinned to:", PINNED_COMMIT)

## 1. What runtime did Colab actually give us?

Colab assigns different GPUs and different disk sizes per session. Nothing
below assumes an A100 or a fixed `/content` size.

In [ ]:
!nvidia-smi
!free -h
!df -h /content

## 2. Mount Drive (transport and persistence only)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
assert os.path.isdir(DRIVE_DATA), f"prepared data not found at {DRIVE_DATA}"
print(sorted(os.listdir(DRIVE_DATA)))

## 3. Clone the repository at the pinned commit

In [ ]:
import subprocess, os

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "fetch", "--all", "--tags"], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--force", PINNED_COMMIT], cwd=REPO_DIR, check=True)

head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR,
                      capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(["git", "status", "--porcelain"], cwd=REPO_DIR,
                       capture_output=True, text=True).stdout.strip()
print("HEAD:", head)
assert not dirty, f"working tree is dirty:\n{dirty}"
os.chdir(REPO_DIR)

## 4. Dependencies

Colab ships PyTorch; only MONAI is normally missing. PyTorch is deliberately
not pinned — the runtime's build matches its own CUDA driver.

In [ ]:
!pip -q install "monai==1.5.1" nibabel

import platform, torch, monai
print(f"python {platform.python_version()}")
print(f"torch  {torch.__version__}  CUDA available: {torch.cuda.is_available()}")
print(f"monai  {monai.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    {props.name}  {props.total_memory/1024**3:.1f} GiB VRAM")
else:
    raise SystemExit("No GPU. Runtime > Change runtime type > GPU.")

## 5. Disk gate — refuse to start what cannot finish

Required space is the extracted cache, plus one shard of staging headroom
(we delete each archive right after extracting it), plus checkpoints and a
safety margin. If `/content` is too small this cell stops.

It does **not** quietly switch to uint8, train from Drive, or use part of the
data. Those would change the experiment.

In [ ]:
import json, shutil, glob, os

meta = json.load(open(f"{DRIVE_DATA}/preprocessing.json"))
shards = sorted(glob.glob(f"{DRIVE_DATA}/shards/*.tar"))
assert shards, "no shards found in Drive"

cache_bytes   = sum(os.path.getsize(s) for s in shards)   # tar ~= sum of npz
largest_shard = max(os.path.getsize(s) for s in shards)
overhead      = 4 * 1024**3           # checkpoints, logs, pip, repo
margin        = 5 * 1024**3
required      = cache_bytes + largest_shard + overhead + margin
free          = shutil.disk_usage("/content").free

GB = 1024**3
print(f"prepared cache      {cache_bytes/GB:7.1f} GiB  ({meta['case_count']} cases, {len(shards)} shards)")
print(f"largest shard       {largest_shard/GB:7.1f} GiB  (staging headroom)")
print(f"checkpoints/logs    {overhead/GB:7.1f} GiB")
print(f"safety margin       {margin/GB:7.1f} GiB")
print(f"REQUIRED            {required/GB:7.1f} GiB")
print(f"free on /content    {free/GB:7.1f} GiB")

if free < required:
    raise SystemExit(
        f"STOP: /content has {free/GB:.1f} GiB but needs {required/GB:.1f} GiB.\n"
        "Use a runtime with a larger disk (Colab Pro high-RAM/A100 runtimes get more),\n"
        "or run this study on a machine with adequate local storage.\n"
        "Do NOT train from /content/drive and do NOT stage a subset."
    )
print("\nOK to stage.")
print(json.dumps(meta, indent=2))

## 6. Stage shards into `/content`, one at a time

For each shard: copy → verify SHA256 → extract → **delete the archive** →
next. Peak disk is therefore *cache + one shard*, not *cache + all archives*.

Re-running is safe: already-extracted shards are skipped.

In [ ]:
import hashlib, os, shutil, subprocess, tarfile, time

os.makedirs(PREPARED_ROOT, exist_ok=True)

sums = {}
for line in open(f"{DRIVE_DATA}/SHA256SUMS"):
    digest, name = line.split()
    sums[os.path.basename(name)] = digest

def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

for source in shards:
    name = os.path.basename(source)
    marker = f"{PREPARED_ROOT}/.staged_{name}"
    if os.path.exists(marker):
        print(f"{name}: already staged")
        continue

    started = time.time()
    local = f"/content/{name}"
    shutil.copyfile(source, local)

    digest = sha256(local)
    if digest != sums[name]:
        os.remove(local)
        raise SystemExit(f"STOP: checksum mismatch for {name}\n  {digest}\n  {sums[name]}")

    with tarfile.open(local) as archive:
        archive.extractall(PREPARED_ROOT)
    os.remove(local)                      # free the archive immediately
    open(marker, "w").close()

    free = shutil.disk_usage("/content").free / 1024**3
    print(f"{name}: verified, extracted, archive removed "
          f"({time.time()-started:5.1f}s, {free:.1f} GiB free)")

for extra in ("manifest.json", "preprocessing.json"):
    shutil.copyfile(f"{DRIVE_DATA}/{extra}", f"{PREPARED_ROOT}/{extra}")
print("\nstaged to", PREPARED_ROOT)

## 7. Verify the staged cache with the repository's own reader

In [ ]:
import sys, glob, json
import numpy as np
sys.path.insert(0, REPO_DIR)
from src.data.prepared import is_case_complete, read_prepared_case

meta = json.load(open(f"{PREPARED_ROOT}/preprocessing.json"))
cases = sorted(glob.glob(f"{PREPARED_ROOT}/cases/*.npz"))
print(f"cases staged: {len(cases)} (expected {meta['case_count']})")
assert len(cases) == meta["case_count"], "incomplete staging - re-run section 6"

ids = [os.path.basename(c)[:-4] for c in cases]
numbers = [int(i.split("_")[1]) for i in ids]
assert len(set(ids)) == len(ids), "duplicate case ids"
assert max(numbers) <= 9000, "a PanTS-te identifier is present"
assert min(os.path.getsize(c) for c in cases) > 0, "a zero-length case file"

for path in cases[:: max(1, len(cases) // 20)]:
    assert is_case_complete(path, deep=True), path
    image, label = read_prepared_case(path)
    assert image.dtype == np.float16 and label.dtype == np.uint8
    assert image.shape == label.shape
    values = image.astype(np.float32)
    assert np.isfinite(values).all() and 0.0 <= values.min() and values.max() <= 1.0
    assert label.max() <= 28

print(f"id range {min(numbers)}..{max(numbers)} — PanTS-tr only")
print("sampled deep validation passed; cache is usable")

## 8. SuPreM checkpoint — download, verify, and prove the transfer

Fetched from the official release rather than a personal copy. The hash was
measured independently; a mismatch stops the notebook.

The transfer report must show **81 of 81** transferable tensors loaded. Only
`conv_final.2.conv.{weight,bias}` stays random: it maps features to class
logits, and SuPreM predicted a different label set, so those two tensors have
the wrong shape and no meaning here.

In [ ]:
import hashlib, os, subprocess

os.makedirs(os.path.dirname(CHECKPOINT), exist_ok=True)
if not os.path.exists(CHECKPOINT):
    subprocess.run(["wget", "-q", "--show-progress", "-O", CHECKPOINT, SUPREM_URL], check=True)

digest = hashlib.sha256(open(CHECKPOINT, "rb").read()).hexdigest()
print(f"size   {os.path.getsize(CHECKPOINT):,} bytes")
print(f"sha256 {digest}")
if digest != SUPREM_SHA256:
    os.remove(CHECKPOINT)
    raise SystemExit(f"STOP: checkpoint hash mismatch, expected {SUPREM_SHA256}")
print("hash verified against the independently measured value\n")

from src.models.segresnet import format_transfer_report, suprem_transfer_report
report = suprem_transfer_report(CHECKPOINT)
print(format_transfer_report(report))
assert report["loaded"] == 81, f"expected 81 transferred tensors, got {report['loaded']}"

## 9. Calibration — measure before spending GPU-hours

Correctness is already established on the laptop. This measures **throughput**
on the GPU Colab actually gave us, at production settings, so the final
schedule is chosen from numbers rather than from a guess.

Run both arms. Do **not** read scientific meaning into these losses.

In [ ]:
import time, torch, subprocess

SHARED = [
    "--prepared-root", PREPARED_ROOT,
    "--manifest", f"{PREPARED_ROOT}/manifest.json",
    "--split", f"{REPO_DIR}/pants_cv_v1.json",
    "--fold", "0",
    "--batch-size", "2", "--samples-per-case", "2", "--accumulation", "2",
    "--num-workers", "2", "--seed", "317",
]

for arm, extra in (("random", []), ("suprem", ["--pretrained-checkpoint", CHECKPOINT])):
    torch.cuda.reset_peak_memory_stats()
    started = time.time()
    subprocess.run(
        ["python", "scripts/train_segresnet.py", "--initialization", arm,
         "--experiment", f"calib_{arm}", "--epochs", "1",
         "--max-steps-per-epoch", "60",
         "--output-root", RUNS_LOCAL, *SHARED, *extra],
        check=True, cwd=REPO_DIR,
    )
    print(f"{arm}: {time.time()-started:.1f}s wall for 60 train steps + validation\n")

!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv
!ls -lh {RUNS_LOCAL}/calib_random/

In [ ]:
# Checkpoint persistence: measure, then choose. ~56 MB per checkpoint.
import os, shutil, time

source = f"{RUNS_LOCAL}/calib_random/latest.pt"
size = os.path.getsize(source) / 1e6
os.makedirs(f"{DRIVE_RUNS}/_timing", exist_ok=True)

started = time.time(); shutil.copyfile(source, f"{RUNS_LOCAL}/_t.pt"); local = time.time() - started
started = time.time(); shutil.copyfile(source, f"{DRIVE_RUNS}/_timing/_t.pt"); to_drive = time.time() - started
os.remove(f"{RUNS_LOCAL}/_t.pt"); os.remove(f"{DRIVE_RUNS}/_timing/_t.pt")

print(f"checkpoint {size:.1f} MB   /content {local:.2f}s   Drive {to_drive:.2f}s")
print("If the Drive write is a few seconds, --output-root can point straight at")
print("Drive (one write per epoch). If it is slow or flaky, keep --output-root on")
print("/content and copy after each epoch with the sync cell in section 12.")

## 10. Production — fold 0, both arms

**Do not run until the schedule below has been agreed.** `EPOCHS` and
`STEPS_PER_EPOCH` are placeholders until calibration numbers are in.

`SHARED_PRODUCTION` is defined once and passed to both arms, which is what
makes the comparison controlled: there is no second place to edit.

In [ ]:
EPOCHS          = None   # <-- set from the calibration report
STEPS_PER_EPOCH = None   # <-- set from the calibration report
assert EPOCHS and STEPS_PER_EPOCH, "agree the schedule from section 9 first"

SHARED_PRODUCTION = [
    "--prepared-root", PREPARED_ROOT,
    "--manifest", f"{PREPARED_ROOT}/manifest.json",
    "--split", f"{REPO_DIR}/pants_cv_v1.json",
    "--fold", "0",
    "--epochs", str(EPOCHS),
    "--max-steps-per-epoch", str(STEPS_PER_EPOCH),
    "--batch-size", "2", "--samples-per-case", "2", "--accumulation", "2",
    "--learning-rate", "1e-4", "--weight-decay", "1e-5",
    "--num-workers", "2", "--seed", "317", "--save-every-epochs", "1",
    "--output-root", RUNS_LOCAL,
]

# One string, used verbatim by BOTH arms. Editing it in one place is what keeps
# the ablation controlled.
SHARED_STRING = " ".join(SHARED_PRODUCTION)
print(SHARED_STRING)

In [ ]:
# ARM A — SegResNet-Random
!cd {REPO_DIR} && python scripts/train_segresnet.py --initialization random --experiment segresnet_random {SHARED_STRING}

In [ ]:
# ARM B — SegResNet initialized from SuPreM supervised pretraining.
# Identical to arm A except --initialization and --pretrained-checkpoint.
!cd {REPO_DIR} && python scripts/train_segresnet.py --initialization suprem --experiment segresnet_suprem --pretrained-checkpoint {CHECKPOINT} {SHARED_STRING}

## 11. RESUME after a Colab disconnect

When the runtime dies, `/content` is destroyed — repo, staged cache, and any
checkpoint not yet copied to Drive. Drive survives.

Recovery: run sections **1 → 8** again in the fresh runtime (same pinned
commit, restage the data), copy the checkpoint back from Drive, then run the
cell below.

**`--epochs` must equal the original run's value.** The cosine schedule stored
in the checkpoint was built for that horizon; resuming with a different number
replays the original curve and the trainer will warn. A real interruption
changes nothing about the configuration.

In [ ]:
EXPERIMENT = "segresnet_random"   # or segresnet_suprem
ARM_ARGS   = ([] if EXPERIMENT.endswith("random")
              else ["--pretrained-checkpoint", CHECKPOINT])

import os, shutil, subprocess, torch
os.makedirs(f"{RUNS_LOCAL}/{EXPERIMENT}", exist_ok=True)
shutil.copyfile(f"{DRIVE_RUNS}/{EXPERIMENT}/latest.pt",
                f"{RUNS_LOCAL}/{EXPERIMENT}/latest.pt")

state = torch.load(f"{RUNS_LOCAL}/{EXPERIMENT}/latest.pt", map_location="cpu",
                   weights_only=False)
provenance = state["config"]
print(f"resuming after epoch {state['epoch']}, global step {state['global_step']}, "
      f"best {state['best_metric']:.4f}")
print(f"  trained at commit {state['git_commit']}")
print(f"  manifest {provenance['manifest_sha256'][:12]}  "
      f"split {provenance['split_sha256'][:12]}")
print(f"  data source {provenance['data_source']}")

# The interrupted run's own epoch budget. Reusing it is not optional: the cosine
# schedule inside the checkpoint was built for this horizon, so changing it here
# would replay the original curve against a different total and silently alter
# the learning-rate trajectory of the second half of training.
original_epochs = provenance["config"]["epochs"]
assert original_epochs == EPOCHS, (
    f"this checkpoint was trained with --epochs {original_epochs}, "
    f"but EPOCHS is {EPOCHS}. Restore the original value before resuming."
)

subprocess.run(
    ["python", "scripts/train_segresnet.py",
     "--initialization", "random" if EXPERIMENT.endswith("random") else "suprem",
     "--experiment", EXPERIMENT,
     "--resume", f"{RUNS_LOCAL}/{EXPERIMENT}/latest.pt",
     *SHARED_PRODUCTION, *ARM_ARGS],
    check=True, cwd=REPO_DIR,
)

## 12. Persist results to Drive

Run after each training cell, and periodically during long runs. Only
`latest.pt`, `best.pt` and the small JSON records are kept — no checkpoint
sprawl.

In [ ]:
import os, shutil, glob

for run in sorted(glob.glob(f"{RUNS_LOCAL}/*")):
    name = os.path.basename(run)
    if name.startswith("_"):
        continue
    destination = f"{DRIVE_RUNS}/{name}"
    os.makedirs(destination, exist_ok=True)
    for artifact in ("latest.pt", "best.pt", "provenance.json", "summary.json"):
        source = f"{run}/{artifact}"
        if os.path.exists(source):
            shutil.copyfile(source, f"{destination}/{artifact}")
    print(f"{name} -> {destination}: {sorted(os.listdir(destination))}")

---

### What survives a disconnect

| | survives | why |
|---|---|---|
| `MyDrive/PanTS_prepared/` | yes | Drive is persistent storage |
| `MyDrive/PanTS_runs/<exp>/latest.pt` | yes | copied by section 12 |
| `/content/PanTS_prepared/` | **no** | ephemeral VM disk — restage from shards |
| `/content/pants_sabin/` | **no** | re-clone at the same pinned commit |
| `/content/runs/` since the last sync | **no** | why section 12 runs per epoch |
| GPU/optimizer state in memory | **no** | rebuilt from `latest.pt` |

Evaluation on PanTS-te happens on the laptop, from the original NIfTIs, after
the model and its postprocessing are frozen. The training cache plays no part
in inference.